In [2]:
# subclassing 모델을 사용 - 데이터 섞기,GradientTape
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [3]:
(x_train, y_train),(x_test,y_test)=tf.keras.datasets.mnist.load_data()
# 구조변경(차원)
print(x_train.shape)  # (60000, 28, 28)
x_train=x_train.reshape((-1,28,28,1)).astype('float32')/255.0    # 타입변경/ 정규화 진행
x_test=x_test.reshape((-1,28,28,1)).astype('float32')/255.0      # 타입변경/ 정규화 진행
print(x_train.shape)  # (60000, 28, 28, 1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(60000, 28, 28, 1)


In [16]:
# tf.data.Dataset.from_tensor_slices 사용하기
import numpy as np
x=np.random.sample((5,2))
print(x)

# dset=tf.data.Dataset.from_tensor_slices(x)
# print(dset)
dset=tf.data.Dataset.from_tensor_slices(x).shuffle(10000).batch(5)
print(dset)
for a in dset:
    print(a)
#----------------------------------

[[0.47297088 0.30482108]
 [0.61479007 0.34948632]
 [0.86971372 0.08702257]
 [0.27195675 0.64165001]
 [0.03128956 0.51356071]]
<_BatchDataset element_spec=TensorSpec(shape=(None, 2), dtype=tf.float64, name=None)>
tf.Tensor(
[[0.86971372 0.08702257]
 [0.27195675 0.64165001]], shape=(2, 2), dtype=float64)
tf.Tensor(
[[0.47297088 0.30482108]
 [0.61479007 0.34948632]], shape=(2, 2), dtype=float64)
tf.Tensor([[0.03128956 0.51356071]], shape=(1, 2), dtype=float64)


In [18]:
# train data 섞기
train_ds=tf.data.Dataset.from_tensor_slices((x_train,y_train)).shuffle(60000).batch(32)
test_ds=tf.data.Dataset.from_tensor_slices((x_test,y_test)).batch(32)
print(train_ds)
print(test_ds)

<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>
<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>


In [25]:
# model
class MyModel(tf.keras.Model):
    def __init__(self):
        super(MyModel,self).__init__()

        self.conv1=tf.keras.layers.Conv2D(filters=32,kernel_size=[3,3],padding='valid',activation='relu')
        self.pool1=tf.keras.layers.MaxPool2D(pool_size=(2,2))

        self.conv2=tf.keras.layers.Conv2D(filters=32,kernel_size=[3,3],padding='valid',activation='relu')
        self.pool2=tf.keras.layers.MaxPool2D(pool_size=(2,2))

        self.flatten=tf.keras.layers.Flatten(dtype='float32')

        self.d1=tf.keras.layers.Dense(units=32,activation='relu')
        self.drop1=tf.keras.layers.Dropout(0.3)
        self.d2=tf.keras.layers.Dense(units=10,activation='softmax')

    def call(self,inputs):    # init에서 선언한 층 호출에 네트워크 구성
        net=self.conv1(inputs)
        net=self.pool1(net)
        net=self.conv2(net)
        net=self.pool2(net)
        net=self.flatten(net)
        net=self.d1(net)
        net=self.drop1(net)
        net=self.d2(net)
        return net

model=MyModel()
temp_inputs=tf.keras.Input(shape=(28,28,1))
model(temp_inputs)

loss_object=tf.keras.losses.SparseCategoricalCrossentropy()
optimizer=tf.keras.optimizers.Adam()
"""
model.compile(optimizer=optimizer,loss=loss_object,metrics=['acc'])
model.fit(x_train,y_train,batch_size=128,epochs=5,verbose=2) # process 기반 스레딩 처리
score=model.evaluate(x_test,y_test)
print('test loss : ',score[0])
print('test acc : ',score[1])
print('예측값 : ',np.argmax(model.predict(x_test[:2]),axis=1))
print('실제값 : ',y_test[:2])
"""

Epoch 1/5
469/469 - 35s - 76ms/step - acc: 0.9065 - loss: 0.3203
Epoch 2/5
469/469 - 40s - 85ms/step - acc: 0.9745 - loss: 0.0844
Epoch 3/5
469/469 - 32s - 68ms/step - acc: 0.9801 - loss: 0.0634
Epoch 4/5
469/469 - 42s - 90ms/step - acc: 0.9846 - loss: 0.0503
Epoch 5/5
469/469 - 42s - 90ms/step - acc: 0.9877 - loss: 0.0414
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - acc: 0.9821 - loss: 0.0490
test loss :  0.040640685707330704
test acc :  0.9858999848365784
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
예측값 :  [7 2]
실제값 :  [7 2]


In [27]:
# GradientTape을 운영 : 모델 서브 프로세싱 학습방법
# 모델 손실과 성능을 측정할 지표 선택. 수집된 측정 지표를 바탕으로 최종결과 출력을 위한 객체 생성
train_loss=tf.keras.metrics.Mean()    # 주어진 값의 (가중)평균을 계산
train_accuracy=tf.keras.metrics.SparseCategoricalAccuracy()
test_loss=tf.keras.metrics.Mean()
test_accuracy=tf.keras.metrics.SparseCategoricalAccuracy()

@tf.function
def train_step(images,labels):  # 얘를 반복하면 loss를 최소화
    with tf.GradientTape() as tape:
        predictions=model(images)
        loss=loss_object(labels,predictions)

    gradients=tape.gradient(loss,model.trainable_variables)   # loss 최소화를 위한 미분 계산
    optimizer.apply_gradients(zip(gradients,model.trainable_variables))
    train_loss(loss)
    train_accuracy(labels,predictions)

@tf.function
def test_step(images,labels):
    predictions=model(images)
    t_loss=loss_object(labels,predictions)
    test_loss(t_loss)
    test_accuracy(labels,predictions)

EPOCHS=5
for epoch in range(EPOCHS):
    for train_images,train_labels in train_ds:
        train_step(train_images,train_labels)

    for test_images,test_labels in train_ds:
        test_step(test_images,test_labels)

    template='epochs:{}, train_loss:{}, train_acc:{},test_loss:{},test_acc:{}'
    print(template.format(epoch+1,train_loss.result(),train_accuracy.result()*100,test_loss.result(),test_accuracy.result()*100))

epochs:1, train_loss:0.03570447489619255, train_acc:98.90999603271484,test_loss:0.02601378783583641,test_acc:99.22666931152344
epochs:2, train_loss:0.03191409632563591, train_acc:99.02832794189453,test_loss:0.023388154804706573,test_acc:99.2933349609375
epochs:3, train_loss:0.02910604327917099, train_acc:99.10610961914062,test_loss:0.02047812007367611,test_acc:99.375
epochs:4, train_loss:0.026684684678912163, train_acc:99.17874908447266,test_loss:0.019651096314191818,test_acc:99.38916778564453
epochs:5, train_loss:0.024709096178412437, train_acc:99.23866271972656,test_loss:0.018526926636695862,test_acc:99.41500091552734


In [28]:
print('예측값 : ',np.argmax(model.predict(x_test[:2]),axis=1))
print('실제값 : ',y_test[:2])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
예측값 :  [7 2]
실제값 :  [7 2]
